# 01 — Exploratory Data Analysis
Run from the repository root. Monetary values are INR crores; TTM is excluded from trends.

In [ ]:
from pathlib import Path
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
root=Path.cwd(); clean=root/'data'/'clean'
pl=pd.read_csv(clean/'profitandloss.csv'); bs=pd.read_csv(clean/'balancesheet.csv'); cf=pd.read_csv(clean/'cashflow.csv'); companies=pd.read_csv(clean/'companies.csv').merge(pd.read_csv(clean/'sector_mapping.csv'),on='symbol',how='left')
numeric=pl.select_dtypes('number'); display(numeric.describe().T); display(pl.isna().mean().sort_values(ascending=False).head(15))
fig,axes=plt.subplots(2,2,figsize=(15,10)); sns.heatmap(pl.isna(),cbar=False,ax=axes[0,0]); axes[0,0].set_title('Profit/loss null map')
latest=pl.loc[~pl.is_ttm.astype(bool)].sort_values('sort_order').groupby('symbol').tail(1).merge(companies[['symbol','sector']],on='symbol',how='left')
sns.histplot(latest.sales,kde=True,ax=axes[0,1]); sns.scatterplot(data=latest,x='sales',y='net_profit',hue='sector',legend=False,ax=axes[1,0]); sns.boxplot(data=latest,x='sector',y='opm_pct',ax=axes[1,1]); axes[1,1].tick_params(axis='x',rotation=90); plt.tight_layout()
display(latest.nlargest(10,'sales')[['symbol','sales','net_profit','opm_pct']]); display(latest.nsmallest(10,'net_profit')[['symbol','sales','net_profit']])
plt.figure(figsize=(10,7)); sns.heatmap(numeric.corr(),cmap='vlag',center=0); plt.title('Correlation matrix'); plt.show()
q1,q3=bs.debt_to_equity.quantile([.25,.75]); iqr=q3-q1; display(bs[(bs.debt_to_equity<q1-1.5*iqr)|(bs.debt_to_equity>q3+1.5*iqr)][['symbol','year_label','debt_to_equity']].sort_values('debt_to_equity',ascending=False).head(20))